In [18]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
load_dotenv("../.env")

engine = create_engine(os.getenv("DATABASE_URL"))

def sql_query(query):
    return pd.read_sql(query, engine)

In [19]:
clase_1 = pd.read_csv("../data/clase_1.csv", sep=";")
clase_2 = pd.read_csv("../data/clase_2.csv", sep=";")
clase_3 = pd.read_csv("../data/clase_3.csv", sep=";")
clase_4 = pd.read_csv("../data/clase_4.csv", sep=";")
claustro = pd.read_csv("../data/claustro.csv", sep=";")

In [20]:
clase_1["Vertical"] = "DS"
clase_2["Vertical"] = "DS"
clase_3["Vertical"] = "FS"
clase_4["Vertical"] = "FS"

alumnos = pd.concat([clase_1, clase_2, clase_3, clase_4], ignore_index=True)
alumnos["Fecha"] = pd.to_datetime(alumnos["Fecha_comienzo"], format="%d/%m/%Y").dt.date

In [21]:
if "Fecha_comienzo" in claustro.columns:
    claustro["Fecha"] = pd.to_datetime(claustro["Fecha_comienzo"], format="%d/%m/%Y").dt.date
else:
    fechas_conocidas = alumnos[["Promoción", "Fecha"]].drop_duplicates()

    meses_repetidos = fechas_conocidas[fechas_conocidas["Promoción"].duplicated(keep=False)]
    if len(meses_repetidos) > 0:
        print(meses_repetidos)
        raise ValueError(
            "Hay varias promociones con el mismo mes. "
            "El claustro necesita una columna de fecha para saber a cuál va cada profesor."
        )

    claustro = claustro.merge(fechas_conocidas, left_on="Promocion", right_on="Promoción", how="left")

In [22]:
campus = pd.concat([alumnos["Campus"], claustro["Campus"]]).drop_duplicates()
df_campus = pd.DataFrame({"nombre_campus": campus})
df_campus

,nombre_campus
0,Madrid
39,Valencia


In [23]:
df_campus.to_sql("campus", con=engine, if_exists="append", index=False)
sql_query("SELECT * FROM campus")

DatabaseError: Execution failed on sql 'INSERT INTO campus (nombre_campus) VALUES (:nombre_campus)': (psycopg.errors.UniqueViolation) duplicate key value violates unique constraint "campus_nombre_campus_key"
DETAIL:  Key (nombre_campus)=(Madrid) already exists.
[SQL: INSERT INTO campus (nombre_campus) VALUES (%(nombre_campus)s::VARCHAR)]
[parameters: [{'nombre_campus': 'Madrid'}, {'nombre_campus': 'Valencia'}]]
(Background on this error at: https://sqlalche.me/e/21/gkpj)

In [ ]:
sql_query("SELECT * FROM campus")

,id_campus,nombre_campus
0,1,Madrid
1,2,Valencia


In [30]:
verticales = pd.concat([alumnos["Vertical"], claustro["Vertical"]]).drop_duplicates()
df_verticales = pd.DataFrame({"nombre_vertical": verticales})
df_verticales

,nombre_vertical
0,DS
25,FS


In [26]:
df_verticales.to_sql("verticales", con=engine, if_exists="append", index=False)
sql_query("SELECT * FROM verticales")

DatabaseError: Execution failed on sql 'INSERT INTO verticales (nombre_vertical) VALUES (:nombre_vertical)': (psycopg.errors.UniqueViolation) duplicate key value violates unique constraint "verticales_nombre_vertical_key"
DETAIL:  Key (nombre_vertical)=(DS) already exists.
[SQL: INSERT INTO verticales (nombre_vertical) VALUES (%(nombre_vertical)s::VARCHAR)]
[parameters: [{'nombre_vertical': 'DS'}, {'nombre_vertical': 'FS'}]]
(Background on this error at: https://sqlalche.me/e/21/gkpj)

In [27]:
modalidades = claustro["Modalidad"].drop_duplicates()
df_modalidad = pd.DataFrame({"nombre_modalidad": modalidades})
df_modalidad

,nombre_modalidad
0,Presencial
7,Online


In [28]:
df_modalidad.to_sql("modalidad", con=engine, if_exists="append", index=False)

DatabaseError: Execution failed on sql 'INSERT INTO modalidad (nombre_modalidad) VALUES (:nombre_modalidad)': (psycopg.errors.UniqueViolation) duplicate key value violates unique constraint "modalidad_nombre_modalidad_key"
DETAIL:  Key (nombre_modalidad)=(Presencial) already exists.
[SQL: INSERT INTO modalidad (nombre_modalidad) VALUES (%(nombre_modalidad)s::VARCHAR)]
[parameters: [{'nombre_modalidad': 'Presencial'}, {'nombre_modalidad': 'Online'}]]
(Background on this error at: https://sqlalche.me/e/21/gkpj)

In [29]:
print(sql_query("SELECT * FROM modalidad"))

   id_modalidad nombre_modalidad
0             1       Presencial
1             2           Online
